System construction and test


In [1]:
from datetime import date, datetime
import pandas as pd
import yfinance as yf
import time
#from Alert import Alert
pd.options.mode.chained_assignment = None  # default='warn'
#alert = Alert('1h','GGAL')

input:
    
     Titulo  : example: GGAL
     frequencia de tick : 1h (frequencia mais alta)
     dftitulo : vista do BD sqtitulosalpha.bd (testar ultimos periodas)
Parametros a variar para back testing ( definir rangos de variação)
     k, d, smooth (parametros do stch)
     dayM  (frequencia media) (multiplicador da frequencia mais alta) exemplo: 5
     semM  (frequencia baixa) (multiplicador da frequencia media) exemplo: 7

In [4]:
dataini = '2013-04-03 19:30:00'
datafim = '2023-04-03 19:30:00'

In [12]:
#%%timeit
import sqlite3
import pandas as pd

# Caminho para o banco de dados
caminho_bd = r'C:\Users\scitr\anaconda_projects\Trading_System\Dados_Fontes\Alpha_Vantage\sqtitulosalpha.db'

# Conectando ao banco
conexao = sqlite3.connect(caminho_bd)

# Lendo a view
#consulta = 'SELECT * FROM vwtitulosdados ORDER BY datetime'
consulta = f"""
SELECT * FROM vwtitulosdados
WHERE datetime BETWEEN '{dataini}' AND '{datafim}'
ORDER BY datetime
"""

dftitulosdados = pd.read_sql_query(consulta, conexao)

# Fechando a conexão
conexao.close()

# Exibindo os primeiros registros para conferir
#display(dftitulosdados)
dftitulosdados = dftitulosdados.drop(columns=["symbol", "moeda", "intervalo","volume"])
#display(len(dftitulosdados))
#display(dftitulosdados.head(10))

In [92]:

i = 'high'
K = 16  
D = 5    
smoth = 5  
dayM = 8  
semM = 10
stpl = 0.02
comission = 0.0035
taxalivrerisgoprom = 0.05


In [94]:
#%%timeit
# Stochastic calculation
def stochastic(df, i, K, D, smoth):
        
    df["k"] = (100. * (df.close - df.low.rolling(K).min()) /
        (df.high.rolling(K).max() - df.low.rolling(K).min()))
    
    df["k" + i] = df.k.rolling(smoth).mean()
    df["d" + i] = df["k" + i].rolling(D).mean()
    
    df.drop(columns=["k"], inplace=True)  

    return df
dfstoch = stochastic(dftitulosdados, i, K, D, smoth)

#display (dfstoch.head(100))
#%time dfstoch

In [99]:
#%%timeit
#pd.set_option('display.max_rows', None)
df = dfstoch_hml
df["longbuylow"] = ((df["klow"] > 20) & (df["klow"] > df["dlow"])).astype(int)
df["longbuymed"] = ((df["kmed"] > 20) & (df["kmed"] > df["dmed"])).astype(int)
df["longbuyhigh"] = ((df["khigh"] > 20) & (df["khigh"] > df["dhigh"])).astype(int)



In [101]:
#%%timeit
def long_buy_crits( df):   
   
    
    df["state"] = "standby"    
    
    
    for i in range(1, len(df)):        

        # long buy  states          

        if  df.loc[i, "longbuyhigh"] == 1 and df.loc[i, "longbuymed"] == 1 and df.loc[i, "longbuylow"] == 1 and df.loc[i-1, "state"] == "standby" :        
            df.loc[i, "state"] = "buylong"

            
            
        if  df.loc[i, "longbuymed"] == 1 and df.loc[i, "longbuylow"] == 1 and (df.loc[i-1, "state"] == "buylong" or df.loc[i-1, "state"] == "staylong") :
            df.loc[i, "state"] = "staylong" 

            
        if  df.loc[i, "longbuyhigh"] == 1 and df.loc[i, "longbuylow"] == 1 and df.loc[i, "longbuymed"] == 0 and (df.loc[i-1, "state"] == "buylong" or df.loc[i-1, "state"] == "staylong") :
            df.loc[i, "state"] = "staylong"

        # long sell state and price 
        
        if  (df.loc[i, "longbuylow"] == 0 ) and  (df.loc[i-1, "state"] == "buylong" or df.loc[i-1, "state"] == "staylong")  :
             df.loc[i, "state"] = "selllong"
            
            
        if  df.loc[i, "longbuyhigh"] == 0 and df.loc[i, "longbuymed"] == 0 and  (df.loc[i-1, "state"] == "buylong" or df.loc[i-1, "state"] == "staylong") :
             df.loc[i, "state"] = "selllong"
             
            
        if  (df.loc[i, "longbuylow"] == 0 or df.loc[i, "longbuymed"] == 0) and  df.loc[i-1, "state"] == "selllong"  :
             df.loc[i, "state"] = "standby" 
      
    return df
    
dfstate = long_buy_crits(dfstoch_hml) 

#%time long_buy_crits(dfstoch_hml)   
#display (df)

In [140]:
dfstate = long_buy_crits(dfstoch_hml)
pd.set_option('display.max_rows', None)
dfbuysell = dfstate[(dfstate['state'] == 'buylong') | (dfstate['state'] == 'selllong')]
#df_intervalo = dfstate.iloc[800:1500] 
#display(df_intervalo)
display (len(dfbuysell))
display(dfbuysell.head(10))

152

,datetime,open,high,low,close,khigh,dhigh,kmed,dmed,klow,dlow,longbuylow,longbuymed,longbuyhigh,state
2094,2014-04-23 12:00:00,10.7955,10.8501,10.7526,10.8345,81.397579,76.312118,69.255456,68.535799,69.520937,54.471845,1,1,1,buylong
2123,2014-04-29 09:00:00,10.2224,10.6084,10.1366,10.5491,19.309208,12.317654,71.756319,71.959015,72.976045,55.556538,1,0,0,selllong
2172,2014-05-07 10:00:00,10.7604,10.7916,10.6435,10.7214,29.906158,26.360437,61.952812,55.822949,77.196191,58.171530,1,1,1,buylong
2200,2014-05-12 14:00:00,10.4018,10.4329,10.2614,10.2926,48.068640,53.379604,55.327496,61.918570,79.640821,60.047888,1,0,0,selllong
2300,2014-05-29 16:00:00,10.1695,10.1695,10.1695,10.1695,84.209052,81.988124,20.019021,16.810004,83.804894,68.088680,1,1,1,buylong
2371,2014-06-11 16:00:00,10.7934,10.7934,10.7934,10.7934,4.885896,12.093813,85.858589,86.728345,87.062151,74.281082,1,0,0,selllong
2434,2014-06-23 15:00:00,11.7838,11.8852,11.6668,11.7604,96.933069,91.091387,36.687909,36.438911,84.245140,78.820115,1,1,1,buylong
2558,2014-07-15 16:00:00,12.5481,12.5481,12.5481,12.5481,16.116236,17.423297,88.161573,88.612559,85.631756,83.660714,1,0,0,selllong
2660,2014-07-31 10:00:00,12.2010,12.6416,12.1035,12.1893,45.850047,41.800198,32.273736,27.120824,85.773616,85.174456,1,1,1,buylong
2691,2014-08-06 09:00:00,10.9337,11.1989,10.7076,10.8869,11.707503,12.602783,40.640336,40.827847,85.559492,85.383346,1,0,0,selllong


In [105]:
#%%timeit
import numpy as np

def long_buy_crits_numpy(df):
    n = len(df)
    state_array = np.full(n, "standby", dtype=object)  # inicializa com "standby"
    estado_anterior = "standby"

    high = df["longbuyhigh"].to_numpy()
    med = df["longbuymed"].to_numpy()
    low = df["longbuylow"].to_numpy()

    for i in range(1, n):
        if high[i] == 1 and med[i] == 1 and low[i] == 1 and estado_anterior == "standby":
            state_array[i] = "buylong"
            estado_anterior = "buylong"
        elif med[i] == 1 and low[i] == 1 and estado_anterior in ["buylong", "staylong"]:
            state_array[i] = "staylong"
            estado_anterior = "staylong"
        elif high[i] == 1 and low[i] == 1 and med[i] == 0 and estado_anterior in ["buylong", "staylong"]:
            state_array[i] = "staylong"
            estado_anterior = "staylong"
        elif low[i] == 0 and estado_anterior in ["buylong", "staylong"]:
            state_array[i] = "selllong"
            estado_anterior = "selllong"
        elif high[i] == 0 and med[i] == 0 and estado_anterior in ["buylong", "staylong"]:
            state_array[i] = "selllong"
            estado_anterior = "selllong"
        elif (low[i] == 0 or med[i] == 0) and estado_anterior == "selllong":
            state_array[i] = "standby"
            estado_anterior = "standby"
        else:
            state_array[i] = estado_anterior

    df["state"] = state_array
    return df

In [107]:
dfstate = long_buy_crits_numpy(dfstoch_hml)
pd.set_option('display.max_rows', None)
dfbuysellop = dfstate[(dfstate['state'] == 'buylong') | (dfstate['state'] == 'selllong')]
#df_intervalo = dfstate.iloc[800:1500] 
#display(df_intervalo)
display (len(dfbuysell))
display(dfbuysell.head(10))

152

,datetime,open,high,low,close,khigh,dhigh,kmed,dmed,klow,dlow,longbuylow,longbuymed,longbuyhigh,state
2094,2014-04-23 12:00:00,10.7955,10.8501,10.7526,10.8345,81.397579,76.312118,69.255456,68.535799,69.520937,54.471845,1,1,1,buylong
2123,2014-04-29 09:00:00,10.2224,10.6084,10.1366,10.5491,19.309208,12.317654,71.756319,71.959015,72.976045,55.556538,1,0,0,selllong
2172,2014-05-07 10:00:00,10.7604,10.7916,10.6435,10.7214,29.906158,26.360437,61.952812,55.822949,77.196191,58.171530,1,1,1,buylong
2200,2014-05-12 14:00:00,10.4018,10.4329,10.2614,10.2926,48.068640,53.379604,55.327496,61.918570,79.640821,60.047888,1,0,0,selllong
2300,2014-05-29 16:00:00,10.1695,10.1695,10.1695,10.1695,84.209052,81.988124,20.019021,16.810004,83.804894,68.088680,1,1,1,buylong
2371,2014-06-11 16:00:00,10.7934,10.7934,10.7934,10.7934,4.885896,12.093813,85.858589,86.728345,87.062151,74.281082,1,0,0,selllong
2434,2014-06-23 15:00:00,11.7838,11.8852,11.6668,11.7604,96.933069,91.091387,36.687909,36.438911,84.245140,78.820115,1,1,1,buylong
2558,2014-07-15 16:00:00,12.5481,12.5481,12.5481,12.5481,16.116236,17.423297,88.161573,88.612559,85.631756,83.660714,1,0,0,selllong
2660,2014-07-31 10:00:00,12.2010,12.6416,12.1035,12.1893,45.850047,41.800198,32.273736,27.120824,85.773616,85.174456,1,1,1,buylong
2691,2014-08-06 09:00:00,10.9337,11.1989,10.7076,10.8869,11.707503,12.602783,40.640336,40.827847,85.559492,85.383346,1,0,0,selllong


In [109]:
#%%timeit
def Stop_Loss_Reentry (dfstate, stpl) :

    df = dfstate[(dfstate['state'] == 'buylong') | (dfstate['state'] == 'staylong')]
    df = df.reset_index(drop=True)
    df = df.drop(columns=["open" ,	"high" , "low" , "khigh","dhigh","kmed" ,"dmed","klow","dlow"])
    #drop(columns=["symbol", "moeda", "intervalo","volume"])
    df["stpl"] = 0.0
    stoplossprice = 0.0
    lastlongbuyprice = 0.0

    for i in range(0, len(df)):
        
        if df.loc[i, "state"] == "buylong" :
           stoplossprice = df.loc[i, "close"]
           df.loc[i,"stpl"] = df.loc[i, "close"] - stoplossprice * (1 - stpl)
           
        if df.loc[i, "state"]== "staylong" :
           df.loc[i, "stpl"] = df.loc[i, "close"] - stoplossprice * (1- stpl)
            
           if df.loc[i, "stpl"] < 0.0 :
                df.loc[i, "state"] = "selllong"
               
           if df.loc[i, "stpl"] > 0.0 and   (df.loc[i-1, "state"] == "selllong" or df.loc[i-1, "state"] == "sellstay") :
                df.loc[i, "state"] = "buylong"
               
           if df.loc[i, "stpl"] < 0.0 and   (df.loc[i-1, "state"] == "selllong" or df.loc[i-1, "state"] == "sellstay") :
                df.loc[i, "state"] = "sellstay"

    return df


In [111]:
pd.set_option('display.max_rows', None)
dflongstpl = Stop_Loss_Reentry (dfstate, stpl)
display(len(dflongstpl))
df_intervalo = dflongstpl.iloc[0:300] 
display(df_intervalo.head(10))

3338

,datetime,close,longbuylow,longbuymed,longbuyhigh,state,stpl
0,2014-04-23 12:00:00,10.8345,1,1,1,buylong,0.21669
1,2014-04-23 13:00:00,10.8384,1,1,1,staylong,0.22059
2,2014-04-23 14:00:00,10.7994,1,1,1,staylong,0.18159
3,2014-04-23 15:00:00,10.7448,1,1,1,staylong,0.12699
4,2014-04-23 16:00:00,10.7604,1,1,0,staylong,0.14259
5,2014-04-24 09:00:00,10.8306,1,1,0,staylong,0.21279
6,2014-04-24 10:00:00,10.9086,1,1,0,staylong,0.29079
7,2014-04-24 11:00:00,10.9710,1,1,0,staylong,0.35319
8,2014-04-24 12:00:00,10.9788,1,1,0,staylong,0.36099
9,2014-04-24 13:00:00,11.1191,1,1,1,staylong,0.50129


In [138]:
dflongsell = dfstate.drop(columns=["open" ,	"high" , "low" , "khigh","dhigh","kmed" ,"dmed","klow","dlow"])
dflongsell = dflongsell[(dflongsell['state'] == 'selllong')]
display (dflongsell.head(10))

dfbuysellstpl = dflongstpl[(dflongstpl['state'] == 'buylong') | (dflongstpl['state'] == 'selllong')]
df_intervalo = dfbuysellstpl.iloc[0:100]
#display(len(dfbuysell))
#display(df_intervalo)


,datetime,close,longbuylow,longbuymed,longbuyhigh,state
2123,2014-04-29 09:00:00,10.5491,1,0,0,selllong
2200,2014-05-12 14:00:00,10.2926,1,0,0,selllong
2371,2014-06-11 16:00:00,10.7934,1,0,0,selllong
2558,2014-07-15 16:00:00,12.5481,1,0,0,selllong
2691,2014-08-06 09:00:00,10.8869,1,0,0,selllong
3525,2015-01-02 16:00:00,12.1789,1,0,0,selllong
3633,2015-01-23 11:00:00,12.6026,1,0,0,selllong
3772,2015-02-18 14:00:00,15.1762,1,0,0,selllong
3836,2015-03-02 14:00:00,16.3304,1,0,0,selllong
3954,2015-03-20 13:00:00,20.2453,1,0,0,selllong


In [115]:
#%%timeit
# Suponha que df1 e df2 têm as mesmas colunas (incluindo 'datetime')
dflongbuysell = pd.concat([dflongsell, dfbuysellstpl], ignore_index=True)

# Ordenar pelo datetime (certifique-se de que é do tipo datetime)
dflongbuysell["datetime"] = pd.to_datetime(dflongbuysell["datetime"])
dflongbuysell = dflongbuysell.sort_values("datetime").reset_index(drop=True)
display (dflongbuysell.head(10))

,datetime,close,longbuylow,longbuymed,longbuyhigh,state,stpl
0,2014-04-23 12:00:00,10.8345,1,1,1,buylong,0.216690
1,2014-04-25 09:00:00,10.5577,1,1,0,selllong,-0.060110
2,2014-04-28 09:00:00,10.6201,1,1,1,buylong,0.002290
3,2014-04-28 10:00:00,10.4953,1,1,1,selllong,-0.122510
4,2014-04-29 09:00:00,10.5491,1,0,0,selllong,NaN
5,2014-05-07 10:00:00,10.7214,1,1,1,buylong,0.214428
6,2014-05-08 14:00:00,10.4485,1,1,0,selllong,-0.058472
7,2014-05-09 15:00:00,10.5499,1,1,1,buylong,0.042928
8,2014-05-12 10:00:00,10.4875,1,0,1,selllong,-0.019472
9,2014-05-12 11:00:00,10.5732,1,0,1,buylong,0.066228


In [117]:
#%%timeit
df = dflongbuysell

# Supondo que seu DataFrame se chame df
cond = (df["state"] == "selllong") & (df["stpl"].isna()) & (df["state"].shift(1) == "selllong")

dflongbuysell = df[~cond].reset_index(drop=True)
display (dflongbuysell.head(10))

,datetime,close,longbuylow,longbuymed,longbuyhigh,state,stpl
0,2014-04-23 12:00:00,10.8345,1,1,1,buylong,0.216690
1,2014-04-25 09:00:00,10.5577,1,1,0,selllong,-0.060110
2,2014-04-28 09:00:00,10.6201,1,1,1,buylong,0.002290
3,2014-04-28 10:00:00,10.4953,1,1,1,selllong,-0.122510
4,2014-05-07 10:00:00,10.7214,1,1,1,buylong,0.214428
5,2014-05-08 14:00:00,10.4485,1,1,0,selllong,-0.058472
6,2014-05-09 15:00:00,10.5499,1,1,1,buylong,0.042928
7,2014-05-12 10:00:00,10.4875,1,0,1,selllong,-0.019472
8,2014-05-12 11:00:00,10.5732,1,0,1,buylong,0.066228
9,2014-05-12 12:00:00,10.4875,1,0,1,selllong,-0.019472


Métricas

In [136]:
df = dflongbuysell
df ["index"] = 100. *(1-comission)
df ["trade"] = 0.
for i in range(1, len(df)):       

                 

    if  df.loc[i, "state"] == "selllong" :
            df.loc[i, "index"] = (((df.loc[i,"close"]-df.loc[i-1,"close"])/df.loc[i-1,"close"])+1) * df.loc[i-1,"index"] * (1-comission)
            df.loc[i, "trade"] = (df.loc[i,"close"]-df.loc[i-1,"close"])/df.loc[i-1,"close"]
    if  df.loc[i, "state"] == "buylong" :
            df.loc[i, "index"] =  df.loc[i-1, "index"] * (1-comission)


display (df.head(10))

,datetime,close,longbuylow,longbuymed,longbuyhigh,state,stpl,index,trade
0,2014-04-23 12:00:00,10.8345,1,1,1,buylong,0.216690,99.650000,0.000000
1,2014-04-25 09:00:00,10.5577,1,1,0,selllong,-0.060110,96.764276,-0.025548
2,2014-04-28 09:00:00,10.6201,1,1,1,buylong,0.002290,96.425601,0.000000
3,2014-04-28 10:00:00,10.4953,1,1,1,selllong,-0.122510,94.958951,-0.011751
4,2014-05-07 10:00:00,10.7214,1,1,1,buylong,0.214428,94.626594,0.000000
5,2014-05-08 14:00:00,10.4485,1,1,0,selllong,-0.058472,91.895228,-0.025454
6,2014-05-09 15:00:00,10.5499,1,1,1,buylong,0.042928,91.573595,0.000000
7,2014-05-12 10:00:00,10.4875,1,0,1,selllong,-0.019472,90.713348,-0.005915
8,2014-05-12 11:00:00,10.5732,1,0,1,buylong,0.066228,90.395852,0.000000
9,2014-05-12 12:00:00,10.4875,1,0,1,selllong,-0.019472,89.349336,-0.008105


In [122]:
def estatisticas_index(df):
    serie = df["index"].dropna()

    maximo = serie.max()
    minimo = serie.min()
    media = serie.mean()
    desvio = serie.std()
    taxagantot = (maximo - minimo)/minimo

    df["datetime"] = pd.to_datetime(df["datetime"])
    data_inicial = df["datetime"].min()
    data_final = df["datetime"].max()
    dias = (data_final - data_inicial).days
    

    taxagananualprom = (1 + taxagantot) ** (1 / dias * 365) - 1  
    taxalivrerisgoprom = 0.05
    sharpe = taxagananualprom/taxalivrerisgoprom
   
    return pd.Series({
        "Máximo": f"{maximo:.2f}",
        "Mínimo ": f"{minimo:.2f}",
        "Média": f"{media:.2f}",
        "Desvio padrão ": f"{desvio:.2f}",
        "Taxa de ganancia total ": f"{taxagantot * 100:.2f}%",
        "Taxa de ganancia anual prom ": f"{taxagananualprom * 100:.2f}%",
        "coef Sharpe": f"{sharpe:.2f}",
    })

In [124]:
resultado_index = estatisticas_index(df)
print(resultado_index.to_string())

Máximo                           107.20
Mínimo                            31.90
Média                             70.32
Desvio padrão                     24.85
Taxa de ganancia total          236.05%
Taxa de ganancia anual prom      14.66%
coef Sharpe                        2.93


In [126]:
def datas_drawdown_max(df):
    df = df.copy()
    df = df[df["index"].notna()]
    df["datetime"] = pd.to_datetime(df["datetime"])

    # Série com índice acumulado
    acumulado = df["index"]
    pico = acumulado.cummax()
    drawdown = acumulado - pico

    # Índice do drawdown máximo
    idx_vale = drawdown.idxmin()
    idx_pico = (acumulado[:idx_vale]).idxmax()
    idx_trademin = df["trade"].idxmin()


    # Datas correspondentes
    data_pico = df.loc[idx_pico, "datetime"]
    data_vale = df.loc[idx_vale, "datetime"]
    data_trademin = df.loc[idx_trademin, "datetime"]

    # Diferença percentual
    valor_pico = df.loc[idx_pico, "index"]
    valor_vale = df.loc[idx_vale, "index"]
    drawdown_pct = ((valor_vale - valor_pico) / valor_pico) * 100
    trademin = df["trade"].min()
    
    return pd.Series({
        "Data do Pico": data_pico.strftime("%Y-%m-%d %H:%M"),
        "Data do Vale": data_vale.strftime("%Y-%m-%d %H:%M"),
        "Valor do Pico": round(valor_pico, 2),
        "Valor do Vale": round(valor_vale, 2),
        "Drawdown Máximo (%)": f"{drawdown_pct:.2f}%",
        "Màxima perdida por trade": f"{df["trade"].min()*100 :.2f}%",
        "Data do Max Loser Trade": data_trademin.strftime("%Y-%m-%d %H:%M"),
    })

In [128]:
resultado_datas = datas_drawdown_max(df)
print(resultado_datas)

Data do Pico                2017-12-13 15:00
Data do Vale                2022-10-19 14:00
Valor do Pico                          107.2
Valor do Vale                           31.9
Drawdown Máximo (%)                  -70.24%
Màxima perdida por trade              -5.69%
Data do Max Loser Trade     2019-03-14 09:00
dtype: object


In [130]:
def estatisticas_trades(df):
    df = df.copy()
    trades = df["trade"].dropna()

    positivos = trades[trades > 0]
    negativos = trades[trades < 0]

    # Porcentagem de positivos
    porcentagem_pos = (len(positivos) / len(trades)) * 100

    resultado = {
        "Total de Trades": len(trades),
        "Percentual de Trades Positivos (%)": f"{porcentagem_pos:.2f}%",
        "Média dos Trades Positivos": round(positivos.mean(), 6),
        "Desvio Padrão (Trades Positivos)": round(positivos.std(), 6),
        "Média dos Trades Negativos": round(negativos.mean(), 6),
        "Desvio Padrão (Trades Negativos)": round(negativos.std(), 6)
    }

    return pd.Series(resultado)

In [134]:
resumo_trades = estatisticas_trades(df)
print(resumo_trades)

Total de Trades                            264
Percentual de Trades Positivos (%)      14.39%
Média dos Trades Positivos            0.053508
Desvio Padrão (Trades Positivos)      0.052189
Média dos Trades Negativos           -0.020524
Desvio Padrão (Trades Negativos)      0.012278
dtype: object


In [148]:
def ranking_drawdowns_puros(df, coluna="index", top_n=5):
    df = df.copy()
    df["datetime"] = pd.to_datetime(df["datetime"])
    serie = df[coluna].dropna().reset_index(drop=True)
    datas = df["datetime"].reset_index(drop=True)

    drawdowns = []

    pico_idx = 0
    pico = serie[0]
    vale_idx = None
    valor_vale = None
    max_dd = 0

    for i in range(1, len(serie)):
        if serie[i] > pico:
            # Se recuperou acima do último pico: salvar ciclo anterior
            if vale_idx is not None and max_dd < 0:
                drawdowns.append({
                    "Data Pico": datas[pico_idx],
                    "Valor Pico": pico,
                    "Data Vale": datas[vale_idx],
                    "Valor Vale": valor_vale,
                    "Drawdown (%)": round(max_dd * 100, 2)
                })

            # Novo pico inicia novo ciclo
            pico = serie[i]
            pico_idx = i
            vale_idx = None
            max_dd = 0
        else:
            dd = (serie[i] - pico) / pico
            if dd < max_dd:
                max_dd = dd
                vale_idx = i
                valor_vale = serie[i]

    # Salva último ciclo, se aplicável
    if vale_idx is not None and max_dd < 0:
        drawdowns.append({
            "Data Pico": datas[pico_idx],
            "Valor Pico": pico,
            "Data Vale": datas[vale_idx],
            "Valor Vale": valor_vale,
            "Drawdown (%)": round(max_dd * 100, 2)
        })

    # Retorna os top N
    df_resultado = pd.DataFrame(drawdowns)
    return df_resultado.sort_values("Drawdown (%)").head(top_n).reset_index(drop=True)

In [150]:
ranking_drawdowns_puros(df, coluna="index", top_n=5)


,Data Pico,Valor Pico,Data Vale,Valor Vale,Drawdown (%)
0,2017-12-13 15:00:00,107.200906,2022-10-19 14:00:00,31.899963,-70.24
1,2015-03-20 13:00:00,102.451331,2017-02-07 12:00:00,82.324758,-19.65
2,2014-04-23 12:00:00,99.650000,2014-12-19 11:00:00,82.404971,-17.31
3,2017-10-11 12:00:00,105.497736,2017-11-20 12:00:00,96.249177,-8.77


In [152]:
def estatisticas_drawdowns_puros(df, coluna="index"):
    df = df.copy()
    df["datetime"] = pd.to_datetime(df["datetime"])
    serie = df[coluna].dropna().reset_index(drop=True)
    datas = df["datetime"].reset_index(drop=True)

    drawdowns = []

    pico_idx = 0
    pico = serie[0]
    vale_idx = None
    valor_vale = None
    max_dd = 0

    for i in range(1, len(serie)):
        if serie[i] > pico:
            if vale_idx is not None and max_dd < 0:
                drawdowns.append(max_dd * 100)  # salva como porcentagem
            # novo ciclo
            pico = serie[i]
            pico_idx = i
            max_dd = 0
            vale_idx = None
        else:
            dd = (serie[i] - pico) / pico
            if dd < max_dd:
                max_dd = dd
                vale_idx = i
                valor_vale = serie[i]

    # salva último ciclo, se houver
    if vale_idx is not None and max_dd < 0:
        drawdowns.append(max_dd * 100)

    # série com drawdowns reais
    serie_dd = pd.Series(drawdowns)

    estatisticas = {
        "Total de Drawdowns": len(drawdowns),
        "Média dos Drawdowns (%)": round(serie_dd.mean(), 2),
        "Desvio Padrão (%)": round(serie_dd.std(), 2),
        "Drawdown Máximo (%)": round(serie_dd.min(), 2),
        "Drawdown Mínimo (%)": round(serie_dd.max(), 2)
    }

    return pd.Series(estatisticas)

In [156]:
resumo_dd_puros = estatisticas_drawdowns_puros(df)
print(resumo_dd_puros)

Total de Drawdowns          4.00
Média dos Drawdowns (%)   -28.99
Desvio Padrão (%)          27.90
Drawdown Máximo (%)       -70.24
Drawdown Mínimo (%)        -8.77
dtype: float64


In [162]:
def estatistica_periodos_estaticos(df, coluna="index"):
    df = df.copy()
    df["datetime"] = pd.to_datetime(df["datetime"])
    df = df[df[coluna].notna()].reset_index(drop=True)

    variacao = df[coluna].diff()
    grupos = (variacao != 0).cumsum()

    agrupado = df.groupby(grupos)
    periodos_estaticos = []

    for _, grupo in agrupado:
        if len(grupo) > 1 and grupo[coluna].nunique() == 1:
            duracao = (grupo["datetime"].iloc[-1] - grupo["datetime"].iloc[0]).total_seconds() / 3600
            periodos_estaticos.append({
                "Valor index": grupo[coluna].iloc[0],
                "Data Início": grupo["datetime"].iloc[0],
                "Data Fim": grupo["datetime"].iloc[-1],
                "Duração (horas)": round(duracao, 2),
                "Número de Registros": len(grupo)
            })

    df_resultado = pd.DataFrame(periodos_estaticos)

    if df_resultado.empty:
        resumo = {
            "Total de Períodos Estáticos": 0,
            "Duração Média (horas)": 0.0,
            "Maior Duração (horas)": 0.0,
            "Data do Maior Período": {"Data Início": None, "Data Fim": None}
        }
    else:
        resumo = {
            "Total de Períodos Estáticos": len(df_resultado),
            "Duração Média (horas)": round(df_resultado["Duração (horas)"].mean(), 2),
            "Maior Duração (horas)": round(df_resultado["Duração (horas)"].max(), 2),
            "Data do Maior Período": df_resultado.loc[df_resultado["Duração (horas)"].idxmax(), ["Data Início", "Data Fim"]].to_dict()
        }

    return df_resultado, pd.Series(resumo)

In [164]:
df_periodos, estatisticas = estatistica_periodos_estaticos(df)
print(df_periodos)
print(estatisticas)

Empty DataFrame
Columns: []
Index: []
Total de Períodos Estáticos                                          0
Duração Média (horas)                                              0.0
Maior Duração (horas)                                              0.0
Data do Maior Período          {'Data Início': None, 'Data Fim': None}
dtype: object
